# Trading Signal Construction from Limit Order Book Data
This notebook demonstrates how to build and evaluate binary classifiers for predicting midprice movement direction using real limit order book (LOB) data. We will load, explore, preprocess, and model the data, then assess performance both in-sample and out-of-sample.

## Loading and Examining Limit Order Book Data
We begin by loading the LOB data and inspecting its structure to ensure it is suitable for analysis and modeling.

### Data Description: Data_A.csv
- **Rows:** 100,000 independent samples, randomly drawn from August–October 2023.
- **Columns:**
    - **Column 1:** Label (midprice change direction: 0 = down, 1 = up).
    - **Columns 2–17:** Price and volume for 4 levels of the LOB (ask/bid sides).
    - **Columns 18–22:** Five previous midprice change directions (0/1 coded).

The data is not a time series; each row is an independent sample.

### Data_B.csv and Notebook Workflow
- **Data_B.csv:** 10,000 independent samples, same structure as Data_A. Used for out-of-sample testing.

**Notebook workflow:**
1. Build and train a binary classifier to predict midprice movement (label) in Data_A.
2. Use the trained classifier to predict labels for Data_B and assess out-of-sample performance.

## Importing Required Packages
We start by importing essential Python libraries for data manipulation, visualization, and file handling.

In [ ]:
# Import essential libraries for data analysis and visualization
import pandas as pd  # Data manipulation
import numpy as np   # Numerical operations
import matplotlib.pyplot as plt  # Plotting
from pathlib import Path  # File path handling

## Reading and Inspecting the Data
We load the CSV file into a pandas DataFrame and print basic information to verify successful loading and understand the data structure.

In [ ]:
# Set the directory containing the data files
data_dir = Path('../data')

In [ ]:
# Read Data_A csv file into pandas DataFrame
file_path = data_dir / 'Data_A.csv'  # Path to Data_A.csv
data = pd.read_csv(file_path, header=None)  # Load data into DataFrame

# Add descriptive column names for easier analysis
data.columns=["label","askl1","vola1","bidl1","volb1","askl2",
            "vola2","bidl2","volb2","askl3","vola3","bidl3","volb3",
            "askl4","vola4","bidl4","volb4",
            "m1","m2","m3","m4","m5"]

# Display the first few rows and print info to check loading
display(data.head())  # Show sample data
print(data.info())    # Print DataFrame info

,label,askl1,vola1,bidl1,volb1,askl2,vola2,bidl2,volb2,askl3,...,volb3,askl4,vola4,bidl4,volb4,m1,m2,m3,m4,m5
0,1,428900.0,1,428700.0,200,429000.0,100,428500.0,300,429100.0,...,300,429200.0,200,428300.0,100,0,1,0,1,0
1,1,427100.0,100,427000.0,100,427200.0,940,426900.0,100,427300.0,...,100,427400.0,700,426700.0,100,0,1,0,1,0
2,1,511300.0,100,511200.0,129,511400.0,500,511100.0,200,511500.0,...,300,511600.0,200,510900.0,300,0,1,1,0,0
3,0,415600.0,200,415400.0,100,415700.0,100,415300.0,100,415800.0,...,600,415900.0,300,415100.0,1020,0,1,1,0,1
4,0,506600.0,300,506500.0,100,506700.0,100,506300.0,200,506800.0,...,1099,506900.0,214,506100.0,699,0,0,1,1,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   label   100000 non-null  int64  
 1   askl1   100000 non-null  float64
 2   vola1   100000 non-null  int64  
 3   bidl1   100000 non-null  float64
 4   volb1   100000 non-null  int64  
 5   askl2   100000 non-null  float64
 6   vola2   100000 non-null  int64  
 7   bidl2   100000 non-null  float64
 8   volb2   100000 non-null  int64  
 9   askl3   100000 non-null  float64
 10  vola3   100000 non-null  int64  
 11  bidl3   100000 non-null  float64
 12  volb3   100000 non-null  int64  
 13  askl4   100000 non-null  float64
 14  vola4   100000 non-null  int64  
 15  bidl4   100000 non-null  float64
 16  volb4   100000 non-null  int64  
 17  m1      100000 non-null  int64  
 18  m2      100000 non-null  int64  
 19  m3      100000 non-null  int64  
 20  m4      100000 non-null  int64  
 21  m5      100

In [ ]:
# Count the occurrences of upward and downward midprice movements
occ = data['label'].value_counts()  # 0 = down, 1 = up
occ  # Display counts

label
1    50267
0    49733
Name: count, dtype: int64

In [ ]:
# Calculate and print the ratio of upward and downward movement cases
ratio_cases = occ / len(data.index)  # Fraction of each label
print(f'Ratio of upward movement cases: {ratio_cases[1]}\nRatio of downward movement cases: {ratio_cases[0]}')

Ratio of upward movement cases: 0.50267
Ratio of downward movement cases: 0.49733


## Splitting Data into Features, Labels, and Validation Set
We separate the label column from the features and split the data into training and validation sets. The validation set is used to assess model generalization and prevent overfitting.

In [ ]:
# Split the data into features (X) and labels (y)
x_train_full = data.drop("label",axis=1)  # Features
y_train_full = np.array(data["label"])   # Labels

# Index for splitting between training and validation data
split_ind = 80000

# Create training and validation sets
x_train = x_train_full[:split_ind]  # Training features
y_train = y_train_full[:split_ind]  # Training labels
x_val = x_train_full[split_ind:]    # Validation features
y_val = y_train_full[split_ind:]    # Validation labels

### Data Inspection: What Should We Be Careful With?
Review the loaded data for potential issues such as class imbalance, missing values, or outliers. These can affect model performance and should be addressed before training.

## Feature Normalization
Features have different scales (e.g., prices vs. volumes). Normalizing ensures that all features contribute equally to model training and prevents issues caused by scale differences.

In [ ]:
# Compute means and standard deviations for each feature from training data
means = x_train.mean(0)  # Mean of each feature
stds = x_train.std(0)   # Standard deviation of each feature

# Normalize training and validation data using training statistics
x_train = (x_train-means)/stds  # Normalized training features
x_val = (x_val-means)/stds      # Normalized validation features

## Training a Logistic Regression Model
We train a logistic regression model to predict midprice movement direction using the normalized features.

### Fitting the Logistic Regression Model
We fit the logistic regression model to the training data and evaluate its accuracy on both training and validation sets.

In [ ]:
# Import logistic regression from scikit-learn
import sklearn
from sklearn.linear_model import LogisticRegression

# Train logistic regression model on training data
reg = LogisticRegression(fit_intercept=True).fit(x_train, y_train)

# Print accuracy on training and validation sets
print('Train Accuracy: ',reg.score(x_train, y_train))
print('Validation Accuracy: ',reg.score(x_val, y_val))

Train Accuracy:  0.7195625
Validation Accuracy:  0.71935


### Model Performance and Next Steps
The logistic regression model achieves about 72% accuracy on both training and validation sets. To improve accuracy, consider adding non-linear features (e.g., squares of features) or trying other modeling approaches.

In [ ]:
# Import polynomial feature transformer and pipeline from scikit-learn
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Create a pipeline that adds polynomial features (degree=2 includes squares and interactions)
poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=True),
    LogisticRegression(fit_intercept=True, solver='newton-cg')
    )

# Fit the quadratic logistic regression model on training data
poly_model.fit(x_train, y_train)

# Print accuracy for training and validation sets
print('Train Accuracy: ', poly_model.score(x_train, y_train))
print('Validation Accuracy: ', poly_model.score(x_val, y_val))

Train Accuracy:  0.735525
Validation Accuracy:  0.7336


### Quadratic Model Results and Expectations
Including quadratic features improves validation accuracy. This suggests the quadratic model will likely outperform the linear model in out-of-sample testing.

## Out-of-Sample Evaluation
We now assess the model's performance on completely unseen data (Data_B.csv) to test its generalization ability.

In [ ]:
# Load and preprocess test data for out-of-sample evaluation
file_path = data_dir / 'Data_B.csv'  # Path to Data_B.csv
df_test = pd.read_csv(file_path, header=None)  # Load test data into DataFrame
df_test.columns=["label","askl1","vola1","bidl1","volb1","askl2",
            "vola2","bidl2","volb2","askl3","vola3","bidl3","volb3",
            "askl4","vola4","bidl4","volb4",
            "m1","m2","m3","m4","m5"]  # Assign column names

# Split into features and labels
x_test = df_test.drop("label",axis=1)  # Test features
y_test = np.array(df_test["label"])    # Test labels

# Normalize test features using training set statistics
x_test = (x_test-means)/stds  # Apply training normalization to test data

In [ ]:
# Assess the out-of-sample accuracy for both models
print('Test Accuracy of Linear Model: ',reg.score(x_test, y_test))  # Linear model accuracy
print('Test Accuracy of Quadratic Model: ',poly_model.score(x_test, y_test))  # Quadratic model accuracy

Test Accuracy of Linear Model:  0.7169
Test Accuracy of Quadratic Model:  0.7339


### Challenge: Can You Further Improve Validation Accuracy?
Consider experimenting with the following strategies:
- **Regularization:** Add L1/L2 penalties to reduce overfitting.
- **Higher-order features:** Try cubic or interaction terms for more complex relationships.
- **Alternative models:** Explore decision trees, random forests, or other classifiers.
Iterative experimentation is key to finding the best model for your data.